In [ ]:
# Day 4: generation and sampling with a small instruct model

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

# Use the GPU if Colab gave us one, otherwise the CPU
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("Running on:", device)

# Download the tokenizer and the model from Hugging Face
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()  # inference mode: we only generate, no training

# Count the model's parameters (all its learned numbers)
n_params = 0
for p in model.parameters():
    n_params += p.numel()

print("Vocab size:", tokenizer.vocab_size)
print("Parameters:", n_params)

Running on: cuda


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Vocab size: 49152
Parameters: 361821120


In [ ]:
# Build the prompt in the chat format the model was trained on (remember SFT!)
messages = [{"role": "user", "content": "What is your name?"}]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("Prompt the model actually sees:")
print(prompt)

# Text -> token ids, as a PyTorch tensor on the same device as the model
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
print("Number of prompt tokens:", input_ids.shape[1])

# Quick check with the library's built-in greedy generation
with torch.no_grad():
    output_ids = model.generate(input_ids, max_new_tokens=30, do_sample=False)

# The output contains the prompt + the new tokens; keep only the new part
new_tokens = output_ids[0][input_ids.shape[1] :]
print("\nModel's answer:", tokenizer.decode(new_tokens, skip_special_tokens=True))

Prompt the model actually sees:
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is your name?<|im_end|>
<|im_start|>assistant

Number of prompt tokens: 35

Model's answer: My name is SmolLM, and I'm a chatbot designed to assist users with their language learning needs. I was trained on a vast amount


In [ ]:
def get_next_token_logits(token_ids):
    """Run the model once on the token ids and return one score per vocab token for the NEXT token."""
    with torch.no_grad():
        outputs = model(token_ids)
    return outputs.logits[0, -1, :]  # last position = prediction for the next token


# Token ids that mean "the reply is finished"
stop_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]


def generate_greedy(prompt_token_ids, max_new_tokens=30):
    """Our own greedy loop: always pick the highest-scoring token. Returns the reply as text."""
    token_ids = prompt_token_ids  # prompt + every token we add
    new_token_ids = []  # only the tokens the model generates

    for step in range(max_new_tokens):
        logits = get_next_token_logits(token_ids)  # 1. scores for every vocab token
        next_token_id = int(torch.argmax(logits))  # 2. greedy: pick the biggest score

        if next_token_id in stop_token_ids:  # 3. stop token? we are done
            break

        new_token_ids.append(next_token_id)
        # 4. add the chosen token to the end of the input and go again
        next_token = torch.tensor([[next_token_id]], device=device)
        token_ids = torch.cat([token_ids, next_token], dim=1)

    return tokenizer.decode(new_token_ids, skip_special_tokens=True)


answer = generate_greedy(input_ids)
print("Our greedy loop  :", answer)
print("Library generate :", tokenizer.decode(new_tokens, skip_special_tokens=True))

Our greedy loop  : My name is SmolLM, and I'm a chatbot designed to assist users with their language learning needs. I was trained on a vast amount
Library generate : My name is SmolLM, and I'm a chatbot designed to assist users with their language learning needs. I was trained on a vast amount


In [ ]:
import random


def roll_dice(probs):
    """Pick an index using the probabilities and one random number between 0 and 1."""
    r = random.random()  # random float, 0 <= r < 1
    running_total = 0.0
    for i in range(len(probs)):
        running_total += float(probs[i])  # end of this token's part of the line
        if r < running_total:
            return i
    return len(probs) - 1  # safety net for tiny rounding errors


def pick_next_token(logits, temperature=1.0, top_k=None, top_p=None, min_p=None):
    """Turn the logits into ONE chosen token id, using temperature, top-k, top-p and min-p."""
    # Greedy: temperature 0 means "always the biggest score"
    if temperature == 0:
        return int(torch.argmax(logits))

    # 1. Temperature: divide the scores before softmax
    logits = logits / temperature

    # 2. Softmax: scores -> probabilities (all add up to 1)
    probs = torch.softmax(logits, dim=-1)

    # 3. Sort from most likely to least likely
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)

    # 4. Top-k: keep only the k most likely tokens
    if top_k is not None:
        sorted_probs = sorted_probs[:top_k]
        sorted_ids = sorted_ids[:top_k]

    # 5. Top-p: keep the smallest set of tokens whose probabilities add up to p
    if top_p is not None:
        keep = 0
        running_total = 0.0
        for p in sorted_probs:
            running_total += float(p)
            keep += 1
            if running_total >= top_p:
                break
        sorted_probs = sorted_probs[:keep]
        sorted_ids = sorted_ids[:keep]

    # 6. Min-p: drop tokens below (min_p x the top token's probability)
    if min_p is not None:
        threshold = min_p * float(sorted_probs[0])
        keep = 0
        for p in sorted_probs:
            if float(p) < threshold:
                break  # sorted, so everything after this is smaller too
            keep += 1
        sorted_probs = sorted_probs[:keep]
        sorted_ids = sorted_ids[:keep]

    # 7. Rescale the kept probabilities so they add up to 1 again
    sorted_probs = sorted_probs / sorted_probs.sum()

    # 8. Roll the dice (your own weighted dice)
    choice = roll_dice(sorted_probs)
    return int(sorted_ids[choice])


def generate(
    prompt_token_ids,
    max_new_tokens=40,
    temperature=1.0,
    top_k=None,
    top_p=None,
    min_p=None,
):
    """Our own generation loop, with any sampling settings."""
    token_ids = prompt_token_ids
    new_token_ids = []

    for step in range(max_new_tokens):
        logits = get_next_token_logits(token_ids)
        next_token_id = pick_next_token(logits, temperature, top_k, top_p, min_p)

        if next_token_id in stop_token_ids:
            break

        new_token_ids.append(next_token_id)
        next_token = torch.tensor([[next_token_id]], device=device)
        token_ids = torch.cat([token_ids, next_token], dim=1)

    return tokenizer.decode(new_token_ids, skip_special_tokens=True)


# Quick check: temperature 0 should give the same answer as greedy
print("temperature=0 :", generate(input_ids, temperature=0))

temperature=0 : My name is SmolLM, and I'm a chatbot designed to assist users with their language learning needs. I was trained on a vast amount of text data, including various languages, and I


In [ ]:
def make_prompt_ids(question):
    """Wrap a question in the chat template and turn it into token ids."""
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return tokenizer(prompt, return_tensors="pt").input_ids.to(device)


moon_ids = make_prompt_ids("Write one short sentence about the moon in brife.")

settings = [
    # --- Temperature only ---
    {"name": "Greedy (T=0)", "temperature": 0, "top_k": None, "top_p": None},
    {"name": "T=0.3", "temperature": 0.3, "top_k": None, "top_p": None},
    {"name": "T=0.7", "temperature": 0.7, "top_k": None, "top_p": None},
    {"name": "T=1.0 (pure sampling)", "temperature": 1.0, "top_k": None, "top_p": None},
    {"name": "T=1.5", "temperature": 1.5, "top_k": None, "top_p": None},
    {"name": "T=2.0", "temperature": 2.0, "top_k": None, "top_p": None},
    # --- Top-k ---
    {"name": "Top-k=1 (T=1)", "temperature": 1.0, "top_k": 1, "top_p": None},
    {"name": "Top-k=10 (T=1)", "temperature": 1.0, "top_k": 10, "top_p": None},
    {"name": "Top-k=50 (T=1)", "temperature": 1.0, "top_k": 50, "top_p": None},
    # --- Top-p ---
    {"name": "Top-p=0.5 (T=1)", "temperature": 1.0, "top_k": None, "top_p": 0.5},
    {"name": "Top-p=0.9 (T=1)", "temperature": 1.0, "top_k": None, "top_p": 0.9},
    # --- Combinations (what real apps use) ---
    {"name": "T=0.7 + Top-p=0.9", "temperature": 0.7, "top_k": None, "top_p": 0.9},
    {"name": "T=1.5 + Top-k=10", "temperature": 1.5, "top_k": 10, "top_p": None},
    {"name": "T=1.5 + Top-p=0.9", "temperature": 1.5, "top_k": None, "top_p": 0.9},
    {
        "name": "T=1.5 + Min-p=0.1",
        "temperature": 1.5,
        "top_k": None,
        "top_p": None,
        "min_p": 0.1,
    },
    {
        "name": "T=2.0 + Min-p=0.1",
        "temperature": 2.0,
        "top_k": None,
        "top_p": None,
        "min_p": 0.1,
    },
]

for s in settings:
    print("=" * 70)
    print(s["name"])
    answers = []
    for run in range(3):
        answer = generate(
            moon_ids,
            max_new_tokens=30,
            temperature=s["temperature"],
            top_k=s["top_k"],
            top_p=s["top_p"],
        )
        answers.append(answer)
        print(f"  {run + 1}. {answer}")

    n_different = len(set(answers))  # set() keeps only the unique answers
    print(f"  -> {n_different} different answers out of 3")

Greedy (T=0)
  1. The moon is a celestial body in our solar system that orbits the Earth, appearing as a bright, natural satellite in the night sky.
  2. The moon is a celestial body in our solar system that orbits the Earth, appearing as a bright, natural satellite in the night sky.
  3. The moon is a celestial body in our solar system that orbits the Earth, appearing as a bright, natural satellite in the night sky.
  -> 1 different answers out of 3
T=0.3
  1. The moon is a celestial body that orbits around the Earth, appearing as a dark, glowing sphere in the night sky.
  2. The moon is a celestial body in our solar system that orbits the Earth, appearing as a bright, natural light in the night sky.
  3. The moon, a celestial body, is a natural satellite orbiting Earth, visible in the night sky as a bright, glowing orb.
  -> 3 different answers out of 3
T=0.7
  1. The moon is a celestial body in our solar system that orbits the Earth, taking approximately one month to complete one or

### What the sampling experiments showed

Prompt: *"Write one short sentence about the moon."* Each setting was run 3 times.

**1. Temperature controls everything.**
- **Low T** sharpens the probabilities, so answers are very similar (T = 0.3 gave three versions of the same sentence).
- **High T** flattens the probabilities. At T = 1.5 and T = 2.0 with no filter, the output was nonsense.
- **T = 0 (greedy) and Top-k = 1** gave the exact same answer every time. Top-k = 1 keeps only the top token, so it is greedy even at T = 1.
- T = 0.7 gave *similar*, not *identical*, answers: over 30 tokens, one different pick snowballs into a different sentence.

**2. Fluent is not the same as true.**
Even readable answers contained made-up facts, e.g. *"the largest natural satellite in the outer Solar System"* (T = 0.7) and *"an invisible body we can't see with the naked eye"* (Top-k = 10). Sampling plus a small model produces confident mistakes, which is hallucination in action.

**3. Top-k tamed high temperature best.**
At T = 1.5, **Top-k = 10** still produced creative, readable sentences. It cuts by *count*, so no matter how flat the probabilities get, only 10 tokens can ever be picked.

**4. Top-p and min-p failed at T = 1.5.**
- **Top-p = 0.9:** after flattening, it needed many low-quality tokens to reach 0.9, so nonsense got in.
- **Min-p = 0.1 (my idea, tested):** also nonsense. In my code, temperature is applied **before** min-p, so the top token's probability is already tiny and the threshold (`0.1 × top`) lets thousands of tokens through.
- Real tools often apply min-p to the **original** probabilities and temperature **last**, which keeps the threshold strict. **The order of samplers matters.**

### Setting → quality / variety → when to use

| Setting | Variety | Quality | Use for |
|---------|---------|---------|---------|
| Greedy (T = 0) / Top-k = 1 | None | Safe, identical every time | Extraction, JSON, classification, tool calls |
| T = 0.3 | Low | High, very similar answers | Factual answers, code |
| Top-p = 0.5 (T = 1) | Low–medium | High | Safe variety |
| T = 0.7 + Top-p = 0.9 | Medium | Fluent, small factual slips | General chat (common default) |
| T = 1.0, no filter | High | Mixed, odd phrases | Rarely on its own |
| T = 1.5 + Top-k = 10 | High | Still readable | Brainstorming, creative text |
| T = 1.5 + Top-p = 0.9 | Very high | Nonsense | Avoid |
| T ≥ 1.5 + Min-p = 0.1 (after temperature) | Very high | Nonsense | Avoid in this order |
| T ≥ 1.5, no filter | Very high | Nonsense | Avoid |

> **Takeaway:** temperature decides how random the model is; filters decide which tokens may be picked, and their order matters. Even fluent output can be false, so agents need low T **and** validation.